In [12]:
import os

# Bloqueia a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

import import_images
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from stardist.plot import render_label
import matplotlib.pyplot as plt

# Carrega o modelo pré-treinado
model = StarDist2D.from_pretrained('2D_versatile_fluo')

# Caminho da imagem
image_path = "/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif"

# Cria um "dicionário temporário" para usar a função carregar_imagem_por_indice
image_paths = {0: image_path}

# Carrega a imagem
image = import_images.carregar_imagem_por_indice(image_paths, 0)
if image is None:
    raise RuntimeError("Falha ao carregar a imagem.")

print(f"Imagem carregada: {image_path} - Dimensão: {image.shape}")

# Predição do modelo
labels, details = model.predict_instances(normalize(image))

# Gera o overlay
overlay = render_label(labels, img=image)

# Cria a pasta de saída
output_dir = os.path.join(os.getcwd(), "resultados", "stardist")
os.makedirs(output_dir, exist_ok=True)

# Nome original da imagem sem extensão
nome_original = os.path.splitext(os.path.basename(image_path))[0]

# ---------- Salva o overlay sozinho ----------
overlay_path = os.path.join(output_dir, f"{nome_original}_overlay.png")
plt.imsave(overlay_path, overlay)
print(f"Overlay salvo em: {overlay_path}")

# ---------- Salva o subplot com input + overlay ----------
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image, cmap="gray")
ax[0].axis("off")
ax[0].set_title("Input image")

ax[1].imshow(overlay)
ax[1].axis("off")
ax[1].set_title("Prediction + input overlay")

subplot_path = os.path.join(output_dir, f"{nome_original}_subplot.png")
plt.savefig(subplot_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Subplot salvo em: {subplot_path}")



Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif - Dimensão: (1024, 1360)
Imagem carregada: /home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/001024-1-001001001.tif - Dimensão: (1024, 1360)
Overlay salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/stardist/001024-1-001001001_overlay.png
Subplot salvo em: /home/kayllany.oliveira/remote-repos/CellViability/resultados/stardist/001024-1-001001001_subplot.png
